Configurations

In [1]:
import yaml
import os
import datetime as dt
import numpy as np
from itertools import product

In [2]:
#- Configuration manager -#

# Create backend default configuration
def get_backend_config(train_config):

    # Get shots from precision
    def get_shots(precision):
        if np.isclose(precision, 0, atol=1e-8):
            return None
        else:
            return int(1/(precision**2))


    # Backend configuration
    backend_config = {
        # Real backend
        'name': "ibm_basquecountry",
        'channel': "ibm_quantum_platform",

        # Noisy backend
        'reset_backend': False, # Get current backend state or load from file
        'timestamp': dt.datetime.now(),

        # Noiseless/Noiseless backend
        'sim_options': {
            'method': 'automatic', # If None, use defalut options (los de abajo)
            'precision': "double", # 'single' or 'double'. Also used to choose torch dtype
            'seed_simulator': train_config['seed']
        },
    }

    # Backend predefined configuration for GPU cuda execution
    if train_config["device"] == "GPU":
        backend_config['sim_options'].update({
            'device': 'GPU', # For nvidia cuda

            'cuStateVec_enable': True,   # NVIDIA library optimization
            'batched_shots_gpu': True,   # Parallelize batch on GPU
            'blocking_enable': False,     # Disable chunking; simulation fits in VRAM 
            #'target_gpus':[0,1], # TODO import torch after? In .py read how many after printing aviables?

            'runtime_parameter_bind_enable': True, # tells Aer to keep the circuit parameterized and bind the numeric values at execution time TODO prueba pa saber las mejores options
        })

    # Backend predefined configuration for noise
    if train_config["execution_type"] in ["noisy", "real"]:
        backend_config['sim_options']['method'] = 'density_matrix'
        backend_config['train_precision'] = 0.015625
        backend_config['eval_precision'] = 0.015625 # Fully TorchConnector: precision cannot be 0 because of Aer SamplerV2
    else:
        backend_config['sim_options']['method'] = 'statevector'
        backend_config['train_precision'] = 0.0
        backend_config['eval_precision'] = 0.0 # Not fully TorchConnector: precision cannot be 0 because of Aer SamplerV2
    backend_config['sim_options']['shots'] = get_shots(backend_config["train_precision"])

    # Eval backend configuration
    backend_config['eval_options'] = backend_config['sim_options'].copy()
    if backend_config['eval_precision'] == 0.0:
        backend_config['eval_options']['method'] = 'statevector'
        backend_config['eval_options']['shots'] = get_shots(backend_config["eval_precision"])


    return backend_config

# Notes
'''
method:
    For noiseless execution:
        - statevector
        - matrix_product_state: more qubits, low entanglement
    For noisy execution:
        - density_matrix
        - statevector: uses less memory
    - stabilizer: only for Clifford circuits (only H, CNOT, and S gates)
    - tensor_network: for large circuits (when reaching memory limits, only GPU)
'''

'''
# More options
backend_for_info = AerSimulator()
print("AerSimulator backend configuration options:")
for option in backend_for_info.options:
    print(" -", option)

'''



# Get data path
def get_data_path(train_config):
    data_path = (
        f"{train_config['data_path']}"
        f"{train_config['implementation']}/"
        f"q{train_config['n_qubits']}/"
        f"{train_config['execution_type']}/"
        f"{train_config['device']}/"
        f"{train_config['gradient_method']}/"
    )
    if 'random_input' in train_config:
        data_path += f"random{train_config['random_input']}/"
    data_path += (
        f"seed{train_config['seed']}/"
        f"id{train_config['run_id']}/"
    )

    return data_path


# Save config file
def create_config_file(train_config, overwrite=False):
    data_path = get_data_path(train_config)
    filename = data_path + "config.yaml"

    fileExists = os.path.exists(filename)
    if fileExists and not overwrite:
        print("Configuration file already exists. Path: " + data_path)
        return filename

    data = {
            'train_config': train_config,
            'backend_config': get_backend_config(train_config)
        }
    
    if not os.path.exists(data_path):
        os.makedirs(data_path)
    
    with open(filename, "w") as file:
        yaml.dump(data, file, default_flow_style=False, sort_keys=False)

    action = "created" if not fileExists else "overwritten"
    print(f"Configuration file {action}. Path: {data_path}")
    return filename



# Load config file
def load_config_file(data_path):
    filename = data_path + "config.yaml"
    
    if not os.path.exists(filename):
        create_config_file(filename)

    with open(filename, "r") as file:
        data = yaml.safe_load(file)

    train_c = data['train_config']
    backend_c = data['backend_config']

    return train_c, backend_c

In [ ]:
#- Configuration manager example -#

# Training configuration dict
train_config = {
    'implementation': "qgan_TorchConnector", # 'qgan_TorchConnector', 'qgan_TorchConnector_ang' or 'qgan_TorchConnector_amp'
    'execution_type': "noiseless", # 'noiseless', 'noisy' or 'real'
    'n_qubits': 4,
    'seed': 0,
    'run_id': 0, # For different circuits or training parameters
    
    # Data
    'data_path': "data/", # Path where all files are stored
    'reset_training_data': False, # Reset training data
    'create_circuits': False, # Create circuits or load from file
    'reset_backend': False, # Get current backend state or load from file

    # Training parameters
    'max_iterations': 1000,
    'gen_iterations': 1,
    'disc_iterations': 3,
    'print_progress_iterations': 1,

    # Compute methods
    'device': "CPU", # 'GPU' or 'CPU'
    'gradient_method': "PSR", # qiskit_algorithms.gradients For now: 'PSR', 'SPSA' and 'REG'

    # Embedding
    'reset_dataset': False, 
    'batch_size': 4, # How many samples' gradients are going to be calculated in a step
    #'real_rate': 0.5, # Rate of training discriminator with real samples instead of generated samples TODO
    'random_input': True, # Add randomness in the input when generating a sample
    'randomness': 0.1 # Multiplier for random values to be less random
}


# Create config file
create_config_file(train_config, overwrite=True)


# Load config file
train_config, backend_config = load_config_file(get_data_path(train_config))


print(train_config)
print(backend_config)

Configuration file overwritten. Path: data/qgan_TorchConnector/q4/noiseless/CPU/PSR/randomTrue/seed0/id0/
{'implementation': 'qgan_TorchConnector', 'execution_type': 'noiseless', 'n_qubits': 4, 'seed': 0, 'run_id': 0, 'data_path': 'data/', 'reset_training_data': False, 'create_circuits': False, 'reset_backend': False, 'max_iterations': 1000, 'gen_iterations': 1, 'disc_iterations': 3, 'print_progress_iterations': 1, 'device': 'CPU', 'gradient_method': 'PSR', 'reset_dataset': False, 'batch_size': 4, 'random_input': True, 'randomness': 0.1}
{'name': 'ibm_basquecountry', 'channel': 'ibm_quantum_platform', 'reset_backend': False, 'timestamp': datetime.datetime(2026, 5, 15, 2, 5, 11, 908364), 'sim_options': {'method': 'statevector', 'precision': 'double', 'seed_simulator': 0, 'shots': None}, 'train_precision': 0.0, 'eval_precision': 0.0, 'eval_options': {'method': 'statevector', 'precision': 'double', 'seed_simulator': 0, 'shots': None}}


In [5]:
#- Create all parameter-combination configuration files -#

# Change these defaults to choose the shared configuration values.
DEFAULT_TRAIN_VALUES = {
    'n_qubits': 4,
    'seed': 0,
    'run_id': 0,
    'data_path': "data/test/",
    'reset_training_data': True,
    'create_circuits': True,
    'reset_backend': True,
    'max_iterations': 2,
    'gen_iterations': 1,
    'disc_iterations': 2,
    'print_progress_iterations': 1,
    'reset_dataset': True,
    'batch_size': 2,
    'randomness': 1,
}

PARAMETER_OPTIONS = {
    'implementation': ("qgan_TorchConnector", "qgan_TorchConnector_ang", "qgan_TorchConnector_amp"),
    'execution_type': ("noiseless", "noisy"), # "real" TODO
    'device': ("CPU",), # TODO gpu
    'gradient_method': ("PSR", "SPSA", "REG"),
    'random_input': (True, False),
}

RANDOM_INPUT_IMPLEMENTATIONS = ("qgan_TorchConnector_ang", "qgan_TorchConnector_amp")
OVERWRITE_CONFIG_FILES = False


def build_train_config_from_options(implementation, execution_type, device, gradient_method, random_input=None):
    train_c = DEFAULT_TRAIN_VALUES.copy()
    train_c.update({
        'implementation': implementation,
        'execution_type': execution_type,
        'device': device,
        'gradient_method': gradient_method,
    })
    if implementation in RANDOM_INPUT_IMPLEMENTATIONS:
        train_c['random_input'] = random_input
    return train_c


created_parameter_combination_files = []
base_options = product(
    PARAMETER_OPTIONS['implementation'],
    PARAMETER_OPTIONS['execution_type'],
    PARAMETER_OPTIONS['device'],
    PARAMETER_OPTIONS['gradient_method'],
)

for implementation, execution_type, device, gradient_method in base_options:
    random_inputs = PARAMETER_OPTIONS['random_input'] if implementation in RANDOM_INPUT_IMPLEMENTATIONS else (None,)
    for random_input in random_inputs:
        train_c = build_train_config_from_options(
            implementation=implementation,
            execution_type=execution_type,
            device=device,
            gradient_method=gradient_method,
            random_input=random_input,
        )
        create_config_file(train_c, overwrite=OVERWRITE_CONFIG_FILES)
        created_parameter_combination_files.append(get_data_path(train_c) + "config.yaml")

print(f"{len(created_parameter_combination_files)} parameter-combination configuration files created.")


Configuration file already exists. Path: data/test/qgan_TorchConnector/q4/noiseless/CPU/PSR/seed0/id0/
Configuration file already exists. Path: data/test/qgan_TorchConnector/q4/noiseless/CPU/SPSA/seed0/id0/
Configuration file already exists. Path: data/test/qgan_TorchConnector/q4/noiseless/CPU/REG/seed0/id0/
Configuration file already exists. Path: data/test/qgan_TorchConnector/q4/noisy/CPU/PSR/seed0/id0/
Configuration file already exists. Path: data/test/qgan_TorchConnector/q4/noisy/CPU/SPSA/seed0/id0/
Configuration file already exists. Path: data/test/qgan_TorchConnector/q4/noisy/CPU/REG/seed0/id0/
Configuration file already exists. Path: data/test/qgan_TorchConnector_ang/q4/noiseless/CPU/PSR/randomTrue/seed0/id0/
Configuration file already exists. Path: data/test/qgan_TorchConnector_ang/q4/noiseless/CPU/PSR/randomFalse/seed0/id0/
Configuration file already exists. Path: data/test/qgan_TorchConnector_ang/q4/noiseless/CPU/SPSA/randomTrue/seed0/id0/
Configuration file already exists. P

In [ ]:
#- Create battery of experiments' configuration files -#

# Change these defaults to choose the shared configuration values.
DEFAULT_TRAIN_VALUES = {
    'seed': 0,
    'run_id': 0,
    'data_path': "data/",
    'reset_training_data': False,
    'create_circuits': False,
    'reset_backend': False,
    'max_iterations': 2,
    'gen_iterations': 1,
    'disc_iterations': 2,
    'print_progress_iterations': 0,
    'reset_dataset': False,
    'batch_size': 2,

    # To change TODO
    'randomness': 0.1,
    'random_input': True,
    'gradient_method': "SPSA",
}

PARAMETER_OPTIONS = {
    'n_qubits': (4, 8, 16),
    'implementation': ("qgan_TorchConnector", "qgan_TorchConnector_ang", "qgan_TorchConnector_amp"),
    'execution_type': ("noiseless", "noisy"), # "real" TODO
    'device': ("CPU", "GPU"),
}

RANDOM_INPUT_IMPLEMENTATIONS = ("qgan_TorchConnector_ang", "qgan_TorchConnector_amp")
OVERWRITE_CONFIG_FILES = False


def build_train_config_from_options(implementation, execution_type, device, gradient_method, random_input=None):
    train_c = DEFAULT_TRAIN_VALUES.copy()
    train_c.update({
        'implementation': implementation,
        'execution_type': execution_type,
        'device': device,
        'gradient_method': gradient_method,
    })
    if implementation in RANDOM_INPUT_IMPLEMENTATIONS:
        train_c['random_input'] = random_input
    return train_c


created_parameter_combination_files = []
base_options = product(
    PARAMETER_OPTIONS['implementation'],
    PARAMETER_OPTIONS['execution_type'],
    PARAMETER_OPTIONS['device'],
    PARAMETER_OPTIONS['gradient_method'],
)

for implementation, execution_type, device, gradient_method in base_options:
    random_inputs = PARAMETER_OPTIONS['random_input'] if implementation in RANDOM_INPUT_IMPLEMENTATIONS else (None,)
    for random_input in random_inputs:
        train_c = build_train_config_from_options(
            implementation=implementation,
            execution_type=execution_type,
            device=device,
            gradient_method=gradient_method,
            random_input=random_input,
        )
        create_config_file(train_c, overwrite=OVERWRITE_CONFIG_FILES)
        created_parameter_combination_files.append(get_data_path(train_c) + "config.yaml")

print(f"{len(created_parameter_combination_files)} parameter-combination configuration files created.")


In [ ]:
#- Execute configuration files -#

from pathlib import Path
import subprocess
import sys


CONFIG_ROOT_PATH = "data/test"
IMPLEMENTATION_NAMES = {"qgan_TorchConnector", "qgan_TorchConnector_ang", "qgan_TorchConnector_amp"}
STOP_ON_ERROR = True
DRY_RUN = False # to print commands without executing


def execute_configs(config_root_path):
    config_root = Path(config_root_path)
    executed_configs = []

    for implementation_dir in sorted(path for path in config_root.iterdir() if path.is_dir()):
        implementation = implementation_dir.name
        if implementation not in IMPLEMENTATION_NAMES:
            continue

        script_path = Path(f"{implementation}.py")
        if not script_path.exists():
            raise FileNotFoundError(f"Implementation script not found: {script_path}")

        for config_path in sorted(implementation_dir.rglob("config.yaml")):
            command = [sys.executable, str(script_path), "-c", str(config_path)]
            print("Executing:", " ".join(command))

            if not DRY_RUN:
                result = subprocess.run(command)
                if result.returncode != 0 and STOP_ON_ERROR:
                    raise RuntimeError(f"Command failed with return code {result.returncode}: {command}")

            executed_configs.append(str(config_path))

    print(f"{len(executed_configs)} configuration files executed.")
    return executed_configs


executed_configs = execute_configs(CONFIG_ROOT_PATH)


Executing: /home/benat/miniconda3/envs/qiskit2/bin/python qgan_TorchConnector.py -c data/test/qgan_TorchConnector/q3/noiseless/CPU/PSR/seed0/id0/config.yaml
Backend file created.
AerSimulator('aer_simulator_statevector')
Circuits file created.
Training data file created.
Training complete: 
   Data path: data/test/qgan_TorchConnector/q3/noiseless/CPU/PSR/seed0/id0/ 
   Best eval: 4.333310977220128 in epoch 0 
   Improvement: 0.0 
   Total time: 1.8531403541564941
Executing: /home/benat/miniconda3/envs/qiskit2/bin/python qgan_TorchConnector.py -c data/test/qgan_TorchConnector/q3/noiseless/CPU/REG/seed0/id0/config.yaml
Backend file created.
AerSimulator('aer_simulator_statevector')
Circuits file created.
Training data file created.
Training complete: 
   Data path: data/test/qgan_TorchConnector/q3/noiseless/CPU/REG/seed0/id0/ 
   Best eval: 4.333310977220128 in epoch 0 
   Improvement: 0.0 
   Total time: 0.6283364295959473
Executing: /home/benat/miniconda3/envs/qiskit2/bin/python qgan_T